# Step 6 — Grant the app service principal, then deploy

This notebook is the fix for the demo's most common failure: an app that deploys fine but **403s at runtime** because its service principal was never granted access to the Genie spaces, Knowledge Assistant, warehouse, and Unity Catalog data.

Order of operations (all driven by `config`):
1. **Create** the app if it doesn't exist — this mints its service principal.
2. **Grant** that SP `CAN_QUERY` / `CAN_RUN` / `CAN_USE` / UC data access on every downstream resource (`grant_app_sp()` from `config`). **This runs before deploy**, so the app is live *and* authorized in one shot.
3. **Sync** `SUPERVISOR_ENDPOINT` from `config` into `app.yaml`.
4. **Deploy** the code.

> Prerequisite: Steps 3 and 5 are done, so `GENIE_SPACE_IDS`, `SUPERVISOR_ENDPOINT`, and `KA_ENDPOINTS` in `config` are filled in.

In [0]:
%run ../config

In [0]:
import os, json
w = workspace_client()

# --- 1. Create the app if missing (mints its service principal) ---
try:
    app = w.apps.get(APP_NAME)
    print(f"App '{APP_NAME}' already exists")
except Exception:
    print(f"Creating app '{APP_NAME}' ...")
    from databricks.sdk.service.apps import App
    w.apps.create_and_wait(app=App(name=APP_NAME,
        description='Multi-source supervisor agent for utility operations (Genie + RAG)'))
    app = w.apps.get(APP_NAME)


### Grant the service principal — BEFORE deploying
This is the step that makes the app work. It grants the app SP access to the supervisor + KA endpoints, the 3 Genie spaces, the warehouse, and the catalog/schema (SELECT cascades to every table + metric view). Idempotent — safe to re-run.

In [0]:
grant_app_sp(APP_NAME)

### Sync `app.yaml` + deploy
Writes the configured `SUPERVISOR_ENDPOINT` into the app's `app.yaml`, syncs the source to the workspace, and deploys.

In [0]:
import os, re

# Locate the 6-app folder that actually contains app.yaml. Robust to both
# standalone execution (this notebook's dir) and %run from the orchestrator
# (the caller's dir = repo root), and to local runs.
def _find_app_dir():
    cands = []
    try:
        nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        d = os.path.dirname(nb)                       # standalone: .../6-app ; %run: repo root
        cands += [d, d + "/6-app", os.path.dirname(d) + "/6-app"]
    except Exception:
        pass
    cands.append(DEMO_ROOT + "/6-app")               # local fallback
    for c in cands:
        for base in ("/Workspace" + c, c):           # workspace FUSE mount, then local
            if os.path.exists(base + "/app.yaml"):
                return c, base                        # (workspace path, filesystem dir)
    raise RuntimeError("Could not find 6-app/app.yaml; checked: " + ", ".join(cands))

APP_WS_PATH, APP_FS_DIR = _find_app_dir()
print("App source folder:", APP_FS_DIR)

# --- 3. Sync SUPERVISOR_ENDPOINT from config into app.yaml ---
yaml_path = APP_FS_DIR + "/app.yaml"
with open(yaml_path) as f:
    y = f.read()
y2 = re.sub(r'(SUPERVISOR_ENDPOINT"\s*\n\s*value:\s*)"[^"]*"', rf'\1"{SUPERVISOR_ENDPOINT}"', y)
if y2 != y:
    open(yaml_path, "w").write(y2)
    print(f"Set SUPERVISOR_ENDPOINT -> {SUPERVISOR_ENDPOINT} in app.yaml")
else:
    print(f"app.yaml already targets {SUPERVISOR_ENDPOINT!r}")
if not SUPERVISOR_ENDPOINT:
    print("NOTE: SUPERVISOR_ENDPOINT is empty - set it in config after Step 5 so the app can call the supervisor.")

In [0]:
# --- 4. Deploy (SDK only; no CLI). The source is already in the workspace, so we
#        deploy directly from the 6-app folder - no sync step needed. ---
from databricks.sdk.service.apps import AppDeployment

source_code_path = APP_FS_DIR if APP_FS_DIR.startswith("/Workspace") else ("/Workspace" + APP_WS_PATH)
print("Deploying from:", source_code_path)
dep = w.apps.deploy_and_wait(app_name=APP_NAME,
        app_deployment=AppDeployment(source_code_path=source_code_path))
print("Deployment state:", dep.status.state if dep.status else "?")
print("App URL:", app_url(APP_NAME))

> The grant in the previous cell ran **before** this deploy, so the app comes up already authorized. If you ever swap the supervisor endpoint, update `SUPERVISOR_ENDPOINT` in `config` and re-run this notebook — it re-grants and redeploys.